# per-residue attribution task

In [1]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import scipy.stats
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from joblib import Parallel, delayed
import os, glob, joblib, gc, esm , torch, ast, numpy as np, pandas as pd




In [2]:
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV,RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, uniform

## select protein-coding gene

In [3]:
gene_name='rpsL'
# pick the reference CDS protein sequence
ref_seqs=pd.read_csv('./data/catalog/protein_sequences.csv')
ref_protein = ref_seqs.loc[ref_seqs['gene'] == gene_name, 'protein_sequence'].values

In [4]:
#load the gene sequence file
gene_sequences=pd.read_csv(f"./data/sequence_data_csv/{gene_name}_combined_sequence_data.csv")
gene_sequences_filtered = gene_sequences[gene_sequences["Frameshift_Mutation"] == 0].copy()
# Ensure required columns exist
if "Protein_Sequence" not in gene_sequences_filtered.columns:
    raise ValueError("CSV file must contain a 'Protein_Sequence' column.")

## load pretrained model

In [10]:
# Assuming Random Forest performed best and saved
import joblib
model = joblib.load(f"./trained_rf_models/{gene_name}_best_rf_model.joblib")

In [16]:


mean_glob   = f"embeddings/{gene_name}/mean/{gene_name}_mean_chunk_*.npz"
out_dir     = f"baseline_preds/{gene_name}_base_preds_chunks"
os.makedirs(out_dir, exist_ok=True)

mean_paths  = sorted(glob.glob(mean_glob))

total = 0
for path in tqdm(mean_paths, desc="baseline preds"):
    z       = np.load(path)                           # identifier, mean_embedding
    X_mean  = z["mean_embedding"]                     # (n_i, 320)
    preds   = model.predict(X_mean).astype("float32")

    chunk_id = os.path.basename(path).split("_")[-1].split(".")[0]
    np.save(f"{out_dir}/base_preds_chunk_{chunk_id}.npy", preds)

    total += len(preds)

print("baseline predictions written for", total, "sequences")


baseline preds: 100%|██████████| 7/7 [00:00<00:00, 35.69it/s]

baseline predictions written for 6542 sequences


In [7]:
import numpy as np, glob

tok_paths = sorted(glob.glob(f"./embeddings/{gene_name}/token/{gene_name}_tok_chunk_*.npz"))

#check all sequences and select max length as seq len
seq_len = 0 
for p in tok_paths: 
    with np.load(p, allow_pickle=True) as z: 
        seq_len = max(seq_len, max(t.shape[0] for t in z["tokens"]))

# embed_dim is always 320 for ESM‑2 T6
embed_dim = 320
print("seq_len =", seq_len)

#sanity check
assert all(
    all(t.shape[0] in {seq_len, seq_len-1} for t in np.load(p, allow_pickle=True)["tokens"])
    for p in tok_paths
), "Found a sequence longer than expected — adjust seq_len!"



seq_len = 124


In [17]:
# ------------------------------------------------------------------
# CONFIG
# # ------------------------------------------------------------------

tok_glob   = f"./embeddings/{gene_name}/token/{gene_name}_tok_chunk_*.npz"
embed_dim  = 320
n_jobs     = 4
BATCH_SIZE = 10  # adjust as needed (10–20 is often good)
model.named_steps["model"].n_jobs = -1  # enable internal RF parallelism

# ------------------------------------------------------------------
# 1) Collect chunk paths
# ------------------------------------------------------------------
tok_paths = sorted(glob.glob(tok_glob))
assert tok_paths, "No *_tok_chunk_*.npz files found"

with np.load(tok_paths[0], mmap_mode="r") as z0:
    seq_len = z0["tokens"].shape[1]
inv = 1.0 / (seq_len - 1)
print(f"L (padded length) = {seq_len}")

# ------------------------------------------------------------------
# 2) Load mean-pools & base predictions
# ------------------------------------------------------------------
mean_pools, base_preds, chunk_sizes = [], [], []
print("loading baseline predictions …")
for tok_p in tqdm(tok_paths):
    with np.load(tok_p, mmap_mode="r") as z:
        toks = z["tokens"]
        mp = toks.mean(axis=1, dtype=np.float32)

    chunk_id = os.path.basename(tok_p).split("_")[-1].split(".")[0]
    pred_path = f"baseline_preds/{gene_name}_base_preds_chunks/base_preds_chunk_{chunk_id}.npy"
    preds = np.load(pred_path, mmap_mode="r")

    mean_pools.append(mp)
    base_preds.append(preds.astype(np.float32))
    chunk_sizes.append(len(preds))

N_total = int(np.sum(chunk_sizes))
print("Total sequences:", N_total)

L (padded length) = 124
loading baseline predictions …


100%|██████████| 7/7 [00:07<00:00,  1.02s/it]

Total sequences: 6542


In [18]:
# ------------------------------------------------------------------
# 3) LOO Importance Calculation (one residue)
# ------------------------------------------------------------------
def importance_for_residue(res_idx: int) -> float:
    abs_sum = 0.0
    for tok_p in tok_paths:
        chunk_id = os.path.basename(tok_p).split("_")[-1].split(".")[0]
        base_path = f"baseline_preds/{gene_name}_base_preds_chunks/base_preds_chunk_{chunk_id}.npy"
        base_preds = np.load(base_path, mmap_mode="r")

        with np.load(tok_p, mmap_mode="r") as z:
            toks = z["tokens"]
            mp   = toks.mean(axis=1, dtype=np.float32)
            rvec = toks[:, res_idx, :]
            masked = (seq_len * mp - rvec) * inv

        preds_masked = model.predict(masked)
        abs_sum += np.abs(base_preds - preds_masked).sum(dtype=np.float64)

        del base_preds, mp, rvec, masked, preds_masked, toks
        gc.collect()

    return abs_sum / N_total


In [19]:

# ------------------------------------------------------------------
# 4) Hybrid: Parallel + Batch‑wise checkpointing
# ------------------------------------------------------------------
out_path = f"residue_importance/{gene_name}_residue_importances_loo.npy"
if os.path.exists(out_path):
    residue_importances = np.load(out_path).tolist()
    start_idx = len(residue_importances)
    print(f"Resuming from residue {start_idx}")
else:
    residue_importances = []
    start_idx = 0

In [ ]:
for start in tqdm(range(start_idx, seq_len, BATCH_SIZE), desc="LOO (hybrid parallel)"):
    end = min(start + BATCH_SIZE, seq_len)

    # Parallel block
    scores = Parallel(n_jobs=12, backend="threading", verbose=0)(
        delayed(importance_for_residue)(idx) for idx in range(start, end)
    )

    residue_importances.extend(scores)
    np.save(out_path, np.asarray(residue_importances, dtype=np.float32))
    print(f"Saved up to residue {end - 1} → {out_path}")

# Final normalization
residue_importances = np.asarray(residue_importances, dtype=np.float32)
residue_importances /= residue_importances.sum()
np.save(out_path, residue_importances)
print("Final normalized importances saved:", out_path)


LOO (hybrid parallel):   8%|▊         | 1/13 [00:26<05:17, 26.46s/it]

Saved up to residue 9 → residue_importance/rpsL_residue_importances_loo.npy


LOO (hybrid parallel):  15%|█▌        | 2/13 [00:48<04:20, 23.65s/it]

Saved up to residue 19 → residue_importance/rpsL_residue_importances_loo.npy


LOO (hybrid parallel):  23%|██▎       | 3/13 [01:10<03:49, 22.95s/it]

Saved up to residue 29 → residue_importance/rpsL_residue_importances_loo.npy


LOO (hybrid parallel):  31%|███       | 4/13 [01:30<03:18, 22.07s/it]

Saved up to residue 39 → residue_importance/rpsL_residue_importances_loo.npy


LOO (hybrid parallel):  38%|███▊      | 5/13 [01:50<02:50, 21.31s/it]

Saved up to residue 49 → residue_importance/rpsL_residue_importances_loo.npy


In [ ]:

# residue_importances = np.load(f"{gene_name}_residue_importances.npy")
# Save residue importance
residue_df = pd.DataFrame({
    "Residue_Position": np.arange(seq_len),
    "Importance": residue_importances
})

residue_df.to_csv(f"residue_importance/{gene_name}_global_residue_importance_all_chunks.csv", index=False)
print("Residue importance saved.")


In [ ]:
# Sort by importance, highest first
residue_df_sorted = residue_df.sort_values("Importance", ascending=False)

# Option A – overwrite the same file
residue_df_sorted.to_csv(
    f"residue_importance/{gene_name}_global_residue_importance_all_chunks.csv", index=False
)

In [ ]:

plt.figure(figsize=(15, 5))
plt.bar(residue_df["Residue_Position"], residue_df["Importance"], color='skyblue')
plt.xlabel("Residue Position")
plt.ylabel("Global Importance Score")
plt.title(f"Global Residue Importance (All Chunks) for {gene_name}")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig(f"residue_importance/{gene_name}_global_residue_importance.png")
plt.show()


In [ ]:
catalog_csv = "./data/filtered_variants_output.csv"                       # ← path to catalogue
TOP_N = 30                                            # how many top residues to inspect

# ------------------------------------------------------------------
# 2)  load & filter WHO catalogue
# ------------------------------------------------------------------
catalog_df = (
    pd.read_csv(catalog_csv)
      .query("gene == @gene_name and Present_R > 0")   # keep only resistant calls
      .copy()
)

# The catalogue uses 1‑based amino‑acid positions (aa_pos).  
# Our residue_importances array used 0‑based indices (0 … 739).
catalog_df["aa_pos_0idx"] = catalog_df["aa_pos"] - 1

print(f"WHO catalogue rows kept: {len(catalog_df)}")

# ------------------------------------------------------------------
# 3)  load residue‑importance scores and take the top‑N
# ------------------------------------------------------------------
imp_df = residue_df_sorted
top_df = imp_df.head(TOP_N)
top_set = set(top_df["Residue_Position"])

print(f"Top‑{TOP_N} model residues selected.")

# ------------------------------------------------------------------
# 4)  intersect catalogue positions with model’s top residues
# ------------------------------------------------------------------
overlap_df = (
    catalog_df[catalog_df["aa_pos_0idx"].isin(top_set)]
      .merge(top_df, left_on="aa_pos_0idx", right_on="Residue_Position")
      .sort_values("Importance", ascending=False)
)

print(f"\nCatalogue residues found in model’s top‑{TOP_N}: {len(overlap_df)}")

# Nice display
display_cols = [
    "Residue_Position", "Importance",
    "variant", "confidence", "Present_R", "Present_S"
]
